### 로컬 LLM

### Ollama

로컬에서 LLM을 쉽게 실행할 수 있게 해주는 도구이다. `ollama pull`로 모델을 다운로드하고 `ollama run`으로 실행하면 된다.

- 공식 사이트: https://ollama.com
- Llama(Meta), Qwen(Alibaba), Gemma(Google) 등 다양한 오픈소스 모델을 지원한다
- 로컬 API 서버를 띄워주기 때문에 LangChain 등 외부 도구에서 쉽게 연동할 수 있다

**양자화**(quantization)는 모델 가중치를 낮은 정밀도로 표현해 모델 크기와 메모리 사용량을 줄이는 기법이다. Ollama에서는 이미 양자화된 모델을 다운로드해 실행할 수 있다.

### Hugging Face

오픈소스 AI 모델의 **GitHub** 같은 플랫폼이다. LLM뿐만 아니라 이미지, 음성 등 다양한 AI 모델이 공유되어 있다.

- 공식 사이트: https://huggingface.co
- **Models**: 수십만 개의 사전학습 모델을 검색하고 다운로드할 수 있다
- **Datasets**: 학습용 데이터셋 공유 및 다운로드
- **Spaces**: 모델을 웹에서 바로 체험할 수 있는 데모 공간

Ollama 라이브러리에 없는 모델도 Hugging Face에서 GGUF 형식으로 다운로드하면 Ollama로 실행할 수 있다. 이처럼 모델을 모아놓은 플랫폼을 **Model Hub**라고 부른다.

### Gemini API 모델로 체인 만들기

프로젝트의 `.env` 파일에 `GOOGLE_API_KEY`를 설정한다.

```dotenv
GOOGLE_API_KEY=발급받은_API_키
```

`load_dotenv()`로 환경 변수를 불러오면 `ChatGoogleGenerativeAI`가 API 키를 사용한다.


In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL_NAME = "gemini-3.6-flash"

prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 {role} 전문가야. 모든 답변은 한국어로 해줘."),
    ("human", "{question}"),
])

llm = ChatGoogleGenerativeAI(model=MODEL_NAME)
parser = StrOutputParser()

chain = prompt | llm | parser

result = chain.invoke({
    "role": "Python",
    "question": "for 문을 설명해줘",
})

print(result)

### Ollama 설치

공식 사이트(https://ollama.com)에서 설치 파일을 다운로드한다.

- **macOS**: `brew install ollama` 또는 공식 사이트에서 다운로드
- **Windows**: 공식 사이트에서 설치 파일 다운로드

설치 후 Ollama 서버를 실행한다. Ollama 앱이 이미 실행 중이면 이 단계는 생략한다.

```bash
ollama serve
```

서버를 실행한 터미널은 열어 둔 상태에서 **새 터미널**을 열어 모델을 다운로드한다.

```bash
ollama pull qwen3.5:0.8b
```

모델명의 `0.8b`는 언어 모델의 파라미터가 약 **8억 개**라는 뜻이다. `B`는 billion(10억)을 의미한다. 파라미터는 모델이 학습 과정에서 조정하는 내부 수치이다.

노트북 실습 중에도 Ollama 서버를 실행한 상태로 유지한다.

```bash
# LangChain Ollama 연동 패키지 설치
pip install langchain-ollama
```

In [ ]:
from langchain_ollama import ChatOllama

# 모델만 교체하면 된다
local_llm = ChatOllama(model="qwen3.5:0.8b", reasoning=False)

# 나머지는 동일
chain = prompt | local_llm | parser

result = chain.invoke({
    "role": "Python",
    "question": "for 문을 설명해줘",
})

print(result)

### 로컬 모델과 Gemini API 비교

| 비교 항목 | 로컬 LLM (qwen3.5:0.8b) | Gemini API (gemini-3.6-flash) |
|-----------|--------------------------|-----------------------------|
| 비용 | API 호출 비용 없음, 장비·전력 필요 | API 요금제와 사용량에 따라 달라짐 |
| 인터넷 | 모델 다운로드 후 로컬 추론에는 불필요 | 필요 |
| 사용 한계 | PC 메모리와 연산 성능의 제약 | 요청 수·토큰 수 등의 API 호출 한도 |